# 第50章 联合分布图（jointplot）

用jointplot同时展示两个变量关系及各自边缘分布。

## 学习目标

本章围绕一种明确的图表结构展开，先看最小可用示例，再加入分组、注释或交互细节。学习重点不是“把图画出来”，而是让图表服务于一个可回答的问题。


## 适用场景

深入检查一对数值变量的联合关系和单变量分布。

## 数据结构

两列连续数值，可增加hue分类。

## 本章练习任务

运行基础图表后，完成以下任务：

1. 将 kind="scatter" 改为 kind="kde" 或 kind="reg"，对比不同中心图类型的信息展示
2. 修改 height 参数从 6 改为 8，观察整体图形尺寸对可读性的影响
3. 调整 ratio 参数（如设为 3），说明主图与边缘图比例对布局的影响


## 0. 准备可复现数据

先完成导入和数据准备，后续单元格只负责一种图表或一种分析动作。


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid", context="notebook")
from js import window
base_url = window.location.origin
diamonds = pd.read_csv(f"{base_url}/datasets/diamonds.csv")
orders_full = diamonds.assign(
    category=diamonds["cut"], channel=diamonds["color"], region=diamonds["clarity"],
    order_value=diamonds["price"], items=diamonds["carat"],
    satisfied=np.where(diamonds["price"] >= diamonds["price"].median(), "高于中位价", "不高于中位价"),
)
orders = orders_full.sample(2_000, random_state=36).copy()
taxis = pd.read_csv(f"{base_url}/datasets/taxis.csv", parse_dates=["pickup", "dropoff"])
marketing_full = taxis.assign(
    channel=taxis["payment"].fillna("unknown"), visits=taxis["distance"],
    ad_spend=taxis["tip"], sales=taxis["total"],
    conversion=(taxis["tip"] / taxis["total"].replace(0, np.nan)).fillna(0),
)
marketing = marketing_full.sample(min(2_000, len(marketing_full)), random_state=36).copy()
flights = pd.read_csv(f"{base_url}/datasets/flights.csv")
daily = flights.assign(
    date=pd.to_datetime(flights["year"].astype("string") + "-" + flights["month"] + "-01"),
    region="AirPassengers", sales=flights["passengers"],
)
print(f"Diamonds：{len(diamonds):,} 行；NYC Taxis：{len(taxis):,} 行；Flights：{len(flights):,} 行")
print("图表兼容列均由公开数据原始字段直接映射；高成本图使用固定 2,000 行样本")


## 1. 基础图表

先保留必要的编码：位置、颜色或大小。图表标题、坐标轴和单位应能让读者脱离代码理解结果。


In [ ]:
grid = sns.jointplot(data=marketing, x="visits", y="sales", kind="scatter", height=6, joint_kws={"alpha": 0.45, "s": 24}, color="#1a73e8")
grid.set_axis_labels("访问量", "销售额")
grid.fig.suptitle("访问量与销售额联合分布", y=1.02)
plt.show()


## 2. 进阶变体

在基础图表可读的前提下增加分组、布局、注释或交互。新增编码必须服务于一个明确问题。


In [ ]:
grid = sns.jointplot(data=marketing, x="visits", y="sales", hue="channel", height=7, palette="colorblind", joint_kws={"alpha": 0.45, "s": 24})
grid.set_axis_labels("访问量", "销售额")
grid.fig.suptitle("分渠道联合分布", y=1.02)
plt.show()


## 3. 参数说明

- kind：scatter/hex/kde/reg/hist
- marginal_kws：边缘设置
- height：尺寸
- ratio：主图比例


## 4. 结果解读

中心图读取关系，边缘图读取各变量分布；两部分应结合解释。


## 常见误区

- 只看中心趋势忽略边缘偏态
- 大样本散点过度重叠
- 不适合一次比较很多变量


## 综合练习

请使用同一份数据完成下面任务，并说明你选择该图表的原因。完成后补充：图表回答了什么问题、最重要的视觉信号是什么、还有哪些信息无法从图中得出。


In [ ]:
grid = sns.jointplot(data=marketing, x="ad_spend", y="conversion", kind="hex", height=6, color="#188038")
grid.set_axis_labels("广告投入", "转化率")
grid.fig.suptitle("广告投入与转化率六边形密度", y=1.02)
plt.show()


## 本章小结

用jointplot同时展示两个变量关系及各自边缘分布。


### 你已经掌握

- 判断联合分布图（jointplot）的适用场景
- 准备与图表匹配的数据结构
- 从基础图表扩展到分组、注释或交互变体
- 按照业务问题解读图表并说明结论边界


### 图表选择速查

| 选择要点 | 本章说明 |
| --- | --- |
| 适用场景 | 深入检查一对数值变量的联合关系和单变量分布。 |
| 数据结构 | 两列连续数值，可增加hue分类。 |
| 结果解读 | 中心图读取关系，边缘图读取各变量分布；两部分应结合解释。 |


### 关键参数

| 参数 | 作用 |
| --- | --- |
| `kind` | scatter/hex/kde/reg/hist |
| `marginal_kws` | 边缘设置 |
| `height` | 尺寸 |
| `ratio` | 主图比例 |


### 需要注意

- 只看中心趋势忽略边缘偏态
- 大样本散点过度重叠
- 不适合一次比较很多变量


### 完成检查

- [ ] 能判断什么问题适合使用联合分布图（jointplot）
- [ ] 能准备符合要求的数据结构
- [ ] 能独立完成基础图表和一个进阶变体
- [ ] 能调整关键参数并解释视觉变化
- [ ] 能根据图表写出有边界的数据结论


### 下一步推荐

把同一图表迁移到另一份数据，先保留同样的编码，再只改变一个维度。比较迁移前后的可读性，并说明哪些结论仍然成立。
